In [ ]:
import polars as pl
df = pl.scan_parquet(
    "/kaggle/input/wikipedia-structured-contents/enwiki/data/enwiki/data/*.parquet"
)

print(df.collect_schema())

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    if files:
        print("  Files:", files[:5])

In [ ]:
df = pl.scan_parquet(
    "/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/*.parquet"
)

print(df.collect_schema())

In [ ]:
sample = df.select([
    "name",
    "url",
    "version",
    "image",
    "infoboxes",
    "sections",
    "references"
]).head(5).collect()

print(sample)

In [ ]:
import json

def count_items(value):
    if value is None:
        return 0
    
    try:
        data = json.loads(value)
        
        if isinstance(data, list):
            return len(data)
        
        return 1
    except:
        return 0


results = df.select([
    "name",
    "url",
    "version",
    "image",
    "infoboxes",
    "sections",
    "references"
]).head(20).collect()

results = results.with_columns([
    pl.col("version")
      .struct.field("number_of_characters")
      .alias("article_length"),

    pl.col("sections")
      .map_elements(count_items, return_dtype=pl.Int64)
      .alias("section_count"),

    pl.col("infoboxes")
      .map_elements(count_items, return_dtype=pl.Int64)
      .alias("infobox_count"),

    pl.col("references")
      .list.len()
      .alias("reference_count")
])

print(
    results.select([
        "name",
        "article_length",
        "section_count",
        "infobox_count",
        "reference_count"
    ])
)

In [ ]:
results = results.with_columns([
    pl.col("reference_count")
      .fill_null(0)
      .cast(pl.Int64),

    pl.when(pl.col("image").is_not_null())
      .then(1)
      .otherwise(0)
      .alias("image_count")
])

print(
    results.select([
        "name",
        "article_length",
        "section_count",
        "infobox_count",
        "image_count",
        "reference_count"
    ])
)

In [ ]:
def count_images(value):
    if value is None:
        return 0

    try:
        data = json.loads(value)

        def search(obj):
            if isinstance(obj, dict):
                count = 0

                if "images" in obj and isinstance(obj["images"], list):
                    count += len(obj["images"])

                for v in obj.values():
                    count += search(v)

                return count

            elif isinstance(obj, list):
                return sum(search(item) for item in obj)

            return 0

        return search(data)

    except:
        return 0


results = results.with_columns([
    pl.col("infobox_count")
      .fill_null(0)
      .cast(pl.Int64),

    pl.col("sections")
      .map_elements(count_images, return_dtype=pl.Int64)
      .alias("image_count")
])

print(
    results.select([
        "name",
        "article_length",
        "section_count",
        "infobox_count",
        "image_count",
        "reference_count"
    ])
)

In [ ]:
def normalize(column):
    minimum = results[column].min()
    maximum = results[column].max()

    if maximum == minimum:
        return pl.lit(0.0)

    return (
        (pl.col(column) - minimum) /
        (maximum - minimum)
    )


results = results.with_columns([
    normalize("article_length").alias("length_score"),
    normalize("section_count").alias("section_score"),
    normalize("infobox_count").alias("infobox_score"),
    normalize("image_count").alias("image_score"),
    normalize("reference_count").alias("reference_score")
])


results = results.with_columns(
    (
        pl.col("length_score") * 0.40 +
        pl.col("section_score") * 0.20 +
        pl.col("infobox_score") * 0.10 +
        pl.col("image_score") * 0.10 +
        pl.col("reference_score") * 0.20
    ).alias("content_richness_score")
)


print(
    results.select([
        "name",
        "article_length",
        "section_count",
        "infobox_count",
        "image_count",
        "reference_count",
        "content_richness_score"
    ]).sort("content_richness_score")
)

In [ ]:
print(results.select(["name", "infoboxes"]).head(5))

In [ ]:
import json

infobox = results["infoboxes"][0]

data = json.loads(infobox)

print(json.dumps(data, indent=2)[:5000])

In [ ]:
def count_infobox_fields(value):
    if value is None:
        return 0

    try:
        data = json.loads(value)

        def search(obj):
            if isinstance(obj, dict):
                count = 1 if obj.get("type") == "field" else 0

                for v in obj.values():
                    count += search(v)

                return count

            elif isinstance(obj, list):
                return sum(search(item) for item in obj)

            return 0

        return search(data)

    except:
        return 0


results = results.with_columns(
    pl.col("infoboxes")
      .map_elements(
          count_infobox_fields,
          return_dtype=pl.Int64
      )
      .alias("infobox_field_count")
)


print(
    results.select([
        "name",
        "infobox_count",
        "infobox_field_count"
    ])
)

In [ ]:
results = results.with_columns(
    pl.col("infobox_field_count")
      .fill_null(0)
      .cast(pl.Int64)
)

print(
    results.select([
        "name",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ])
)

In [ ]:
def normalize(column):
    minimum = results[column].min()
    maximum = results[column].max()

    if maximum == minimum:
        return pl.lit(0.0)

    return (
        (pl.col(column) - minimum) /
        (maximum - minimum)
    )


results = results.with_columns([
    normalize("article_length").alias("length_score"),
    normalize("section_count").alias("section_score"),
    normalize("infobox_field_count").alias("infobox_score"),
    normalize("image_count").alias("image_score"),
    normalize("reference_count").alias("reference_score")
])


results = results.with_columns(
    (
        pl.col("length_score") * 0.40 +
        pl.col("section_score") * 0.20 +
        pl.col("infobox_score") * 0.10 +
        pl.col("image_score") * 0.10 +
        pl.col("reference_score") * 0.20
    ).alias("content_richness_score")
)


print(
    results.select([
        "name",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count",
        "content_richness_score"
    ]).sort("content_richness_score")
)

In [ ]:
ranked_results = (
    results
    .sort("content_richness_score")
    .with_row_index("rank", offset=1)
)

print(
    ranked_results.select([
        "rank",
        "name",
        "content_richness_score",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ])
)

In [ ]:
large_sample = (
    df
    .select([
        "name",
        "url",
        "version",
        "image",
        "infoboxes",
        "sections",
        "references"
    ])
    .head(10000)
    .collect()
)

print(large_sample.shape)

In [ ]:
large_sample = large_sample.with_columns([
    pl.col("version")
      .struct.field("number_of_characters")
      .alias("article_length"),

    pl.col("sections")
      .map_elements(count_items, return_dtype=pl.Int64)
      .alias("section_count"),

    pl.col("infoboxes")
      .map_elements(count_infobox_fields, return_dtype=pl.Int64)
      .fill_null(0)
      .alias("infobox_field_count"),

    pl.col("sections")
      .map_elements(count_images, return_dtype=pl.Int64)
      .alias("image_count"),

    pl.col("references")
      .list.len()
      .fill_null(0)
      .cast(pl.Int64)
      .alias("reference_count")
])

print(
    large_sample.select([
        "name",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ]).head(10)
)

In [ ]:
def normalize_large(column):
    minimum = large_sample[column].min()
    maximum = large_sample[column].max()

    if maximum == minimum:
        return pl.lit(0.0)

    return (
        (pl.col(column) - minimum) /
        (maximum - minimum)
    )


large_sample = large_sample.with_columns([
    normalize_large("article_length").alias("length_score"),
    normalize_large("section_count").alias("section_score"),
    normalize_large("infobox_field_count").alias("infobox_score"),
    normalize_large("image_count").alias("image_score"),
    normalize_large("reference_count").alias("reference_score")
])


large_sample = large_sample.with_columns(
    (
        pl.col("length_score") * 0.40 +
        pl.col("section_score") * 0.20 +
        pl.col("infobox_score") * 0.10 +
        pl.col("image_score") * 0.10 +
        pl.col("reference_score") * 0.20
    ).alias("content_richness_score")
)


print(
    large_sample.select([
        "name",
        "content_richness_score",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ])
    .sort("content_richness_score")
    .head(20)
)

In [ ]:
large_sample = large_sample.with_columns(
    (pl.col("content_richness_score") * 100)
    .round(2)
    .alias("content_richness_score_100")
)

print(
    large_sample.select([
        "name",
        "content_richness_score_100",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ])
    .sort("content_richness_score_100")
    .head(20)
)

In [ ]:
large_sample = large_sample.with_columns([
    pl.col("article_length").rank(pct=True).alias("length_percentile"),
    pl.col("section_count").rank(pct=True).alias("section_percentile"),
    pl.col("infobox_field_count").rank(pct=True).alias("infobox_percentile"),
    pl.col("image_count").rank(pct=True).alias("image_percentile"),
    pl.col("reference_count").rank(pct=True).alias("reference_percentile")
])

print(
    large_sample.select([
        "name",
        "length_percentile",
        "section_percentile",
        "infobox_percentile",
        "image_percentile",
        "reference_percentile"
    ]).head(10)
)

In [ ]:
large_sample = large_sample.with_columns([
    (
        pl.col("article_length").rank()
        / pl.len()
    ).alias("length_percentile"),

    (
        pl.col("section_count").rank()
        / pl.len()
    ).alias("section_percentile"),

    (
        pl.col("infobox_field_count").rank()
        / pl.len()
    ).alias("infobox_percentile"),

    (
        pl.col("image_count").rank()
        / pl.len()
    ).alias("image_percentile"),

    (
        pl.col("reference_count").rank()
        / pl.len()
    ).alias("reference_percentile")
])

print(
    large_sample.select([
        "name",
        "length_percentile",
        "section_percentile",
        "infobox_percentile",
        "image_percentile",
        "reference_percentile"
    ]).head(10)
)

In [ ]:
large_sample = large_sample.with_columns(
    (
        pl.col("length_percentile") * 0.40 +
        pl.col("section_percentile") * 0.20 +
        pl.col("infobox_percentile") * 0.10 +
        pl.col("image_percentile") * 0.10 +
        pl.col("reference_percentile") * 0.20
    )
    .mul(100)
    .round(2)
    .alias("content_richness_score")
)

print(
    large_sample.select([
        "name",
        "content_richness_score",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ])
    .sort("content_richness_score")
    .head(20)
)

In [ ]:
final_ranking = (
    large_sample
    .sort("content_richness_score")
    .with_row_index("rank", offset=1)
)

print(
    final_ranking.select([
        "rank",
        "name",
        "content_richness_score",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ])
    .head(20)
)

In [ ]:
random_sample = (
    df
    .select([
        "name",
        "url",
        "version",
        "image",
        "infoboxes",
        "sections",
        "references"
    ])
    .sample(n=10000, seed=42)
    .collect()
)

print(random_sample.shape)

In [ ]:
random_sample = (
    df
    .select([
        "name",
        "url",
        "version",
        "image",
        "infoboxes",
        "sections",
        "references"
    ])
    .collect()
    .sample(n=10000, seed=42)
)

print(random_sample.shape)

In [ ]:
import polars as pl

df = pl.scan_parquet(
    "/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/*.parquet"
)

print(df.collect_schema())

In [ ]:
import glob
import polars as pl

files = glob.glob(
    "/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/*.parquet"
)

columns = [
    "name",
    "url",
    "version",
    "image",
    "infoboxes",
    "sections",
    "references"
]

parts = []

for file in files:
    part = pl.read_parquet(
        file,
        columns=columns,
        n_rows=150
    )
    parts.append(part)

safe_sample = (
    pl.concat(parts)
    .sample(n=10000, seed=42)
)

print("Articles:", safe_sample.height)
print("Columns:", safe_sample.width)

In [ ]:
import json

def count_items(value):
    if value is None:
        return 0

    try:
        data = json.loads(value)

        if isinstance(data, list):
            return len(data)

        return 1
    except:
        return 0


def count_images(value):
    if value is None:
        return 0

    try:
        data = json.loads(value)

        def search(obj):
            if isinstance(obj, dict):
                count = 0

                if "images" in obj and isinstance(obj["images"], list):
                    count += len(obj["images"])

                for v in obj.values():
                    count += search(v)

                return count

            elif isinstance(obj, list):
                return sum(search(item) for item in obj)

            return 0

        return search(data)

    except:
        return 0


def count_infobox_fields(value):
    if value is None:
        return 0

    try:
        data = json.loads(value)

        def search(obj):
            if isinstance(obj, dict):
                count = 1 if obj.get("type") == "field" else 0

                for v in obj.values():
                    count += search(v)

                return count

            elif isinstance(obj, list):
                return sum(search(item) for item in obj)

            return 0

        return search(data)

    except:
        return 0

print("Feature functions ready!")

In [ ]:
safe_sample = safe_sample.with_columns([
    pl.col("version")
      .struct.field("number_of_characters")
      .alias("article_length"),

    pl.col("sections")
      .map_elements(count_items, return_dtype=pl.Int64)
      .alias("section_count"),

    pl.col("infoboxes")
      .map_elements(count_infobox_fields, return_dtype=pl.Int64)
      .fill_null(0)
      .alias("infobox_field_count"),

    pl.col("sections")
      .map_elements(count_images, return_dtype=pl.Int64)
      .alias("image_count"),

    pl.col("references")
      .list.len()
      .fill_null(0)
      .cast(pl.Int64)
      .alias("reference_count")
])

print(
    safe_sample.select([
        "name",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ]).head(10)
)

In [ ]:
safe_sample = safe_sample.with_columns([
    (
        pl.col("article_length").rank() / pl.len()
    ).alias("length_percentile"),

    (
        pl.col("section_count").rank() / pl.len()
    ).alias("section_percentile"),

    (
        pl.col("infobox_field_count").rank() / pl.len()
    ).alias("infobox_percentile"),

    (
        pl.col("image_count").rank() / pl.len()
    ).alias("image_percentile"),

    (
        pl.col("reference_count").rank() / pl.len()
    ).alias("reference_percentile")
])

print(
    safe_sample.select([
        "name",
        "length_percentile",
        "section_percentile",
        "infobox_percentile",
        "image_percentile",
        "reference_percentile"
    ]).head(10)
)

In [ ]:
safe_sample = safe_sample.with_columns(
    (
        pl.col("length_percentile") * 0.40 +
        pl.col("section_percentile") * 0.20 +
        pl.col("infobox_percentile") * 0.10 +
        pl.col("image_percentile") * 0.10 +
        pl.col("reference_percentile") * 0.20
    )
    .mul(100)
    .round(2)
    .alias("content_richness_score")
)

print(
    safe_sample.select([
        "name",
        "content_richness_score",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ])
    .sort("content_richness_score")
    .head(20)
)

In [ ]:
final_ranking = (
    safe_sample
    .sort("content_richness_score")
    .with_row_index("rank", offset=1)
)

print(
    final_ranking.select([
        "rank",
        "name",
        "content_richness_score",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ]).head(20)
)

In [ ]:
threshold = safe_sample["content_richness_score"].quantile(0.05)

limited_content = (
    safe_sample
    .filter(pl.col("content_richness_score") <= threshold)
    .sort("content_richness_score")
)

print(f"5% score threshold: {threshold:.2f}")
print(f"Articles identified: {limited_content.height}")

print(
    limited_content.select([
        "name",
        "content_richness_score",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ]).head(20)
)

In [ ]:
print("===== WikiWeak Article Finder =====")
print(f"Articles analyzed: {safe_sample.height:,}")
print(f"Limited-content threshold: {threshold:.2f}/100")
print(f"Articles identified in bottom 5%: {limited_content.height:,}")

print("\nScoring factors:")
print("• Article length       → 40%")
print("• Section count        → 20%")
print("• Infobox fields       → 10%")
print("• Images               → 10%")
print("• References           → 20%")

print("\nScore interpretation:")
print("The content-richness score is an experimental relative measure")
print("based on percentile ranks within the analyzed sample.")
print("It is not an official Wikimedia quality rating.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(
    safe_sample["content_richness_score"].to_list(),
    bins=30
)

plt.xlabel("Content-Richness Score (0–100)")
plt.ylabel("Number of Articles")
plt.title("Distribution of Content-Richness Scores")

plt.show()

In [ ]:
top20 = limited_content.sort("content_richness_score").head(20)

plt.figure(figsize=(10, 7))

plt.barh(
    top20["name"].to_list()[::-1],
    top20["content_richness_score"].to_list()[::-1]
)

plt.xlabel("Content-Richness Score")
plt.ylabel("Article")
plt.title("20 Relatively Limited-Content Articles")

plt.tight_layout()
plt.show()

In [ ]:
# Select one example article
example = limited_content.head(1)

print("===== Score Breakdown =====")

print("Article:", example["name"][0])

print("\nFactors:")

print("Article Length:", example["article_length"][0])
print("Sections:", example["section_count"][0])
print("Infobox Fields:", example["infobox_field_count"][0])
print("Images:", example["image_count"][0])
print("References:", example["reference_count"][0])

print("\nFinal Score:", example["content_richness_score"][0])

In [ ]:
final_results = (
    limited_content
    .select([
        "name",
        "url",
        "content_richness_score",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ])
    .sort("content_richness_score")
)

print("===== WikiWeak Results =====")
print(final_results.head(20))



# WikiWeak Article Finder

## Project Overview

WikiWeak Article Finder analyzes Wikipedia articles from the Wikimedia Structured Contents dataset and identifies articles with relatively limited content based on measurable content factors.

### Factors Used

| Factor | Weight |
|---|---:|
| Article Length | 40% |
| Section Count | 20% |
| Infobox Fields | 10% |
| Images | 10% |
| References | 20% |

### Scoring Method

Each factor is converted into a percentile rank within the analyzed sample.

The final Content-Richness Score is calculated as:

**Score = (Length × 40%) + (Sections × 20%) + (Infobox × 10%) + (Images × 10%) + (References × 20%)**

The result is expressed on a 0–100 scale.

### Results

- **Articles analyzed:** 10,000
- **Limited-content threshold:** 11.76
- **Articles identified in bottom 5%:** 502

The score is an experimental relative measure for this project. It is **not an official Wikimedia quality rating**.

### Output

The project provides:

1. Content-richness scores for analyzed articles
2. Ranking of articles by score
3. Identification of relatively limited-content articles
4. Score distribution visualization
5. Top 20 limited-content article visualization
6. Individual score breakdown


# WikiWeak Article Finder

## Project Overview

WikiWeak Article Finder analyzes Wikipedia articles from the Wikimedia Structured Contents dataset and identifies articles with relatively limited content based on measurable content factors.

### Factors Used

- Article Length: 40%
- Section Count: 20%
- Infobox Fields: 10%
- Images: 10%
- References: 20%
### Scoring Method

Each factor is converted into a percentile rank within the analyzed sample.

The final Content-Richness Score is calculated as:

**Score = (Length × 40%) + (Sections × 20%) + (Infobox × 10%) + (Images × 10%) + (References × 20%)**

The result is expressed on a 0–100 scale.

### Results

- **Articles analyzed:** 10,000
- **Limited-content threshold:** 11.76
- **Articles identified in bottom 5%:** 502

The score is an experimental relative measure for this project. It is **not an official Wikimedia quality rating**.

### Output

The project provides:

1. Content-richness scores for analyzed articles
2. Ranking of articles by score
3. Identification of relatively limited-content articles
4. Score distribution visualization
5. Top 20 limited-content article visualization
6. Individual score breakdown


i